# Lesson 01: Introduction to AI Engineering

## Learning Objectives
- Understand what AI Engineering is and how it differs from traditional ML engineering
- Learn about Foundation Models and their evolution
- Make your first API call to experience a large language model firsthand
- Compare the response styles and capabilities of different models

> This notebook is based on Chip Huyen's *AI Engineering: Building Applications with Foundation Models*.
> All code cells include detailed comments, suitable for learners with no programming experience.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [8]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'deepseek'  # 'openai' / 'deepseek' / 'openrouter' / 'ollama'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


Connected! provider = deepseek, default model = deepseek-v4-flash


---

## Activity 1: Your First "Chat" with AI (via Code)

### Activity Goal
Use Python to call a GPT model and have it help you with a simple task. This gives you a feel for "controlling AI with code."

It's like chatting with ChatGPT in the browser — except this time you are sending messages from code.

In [9]:
# Activity 1: Ask AI to write an email

# Define the message you want to send to the AI
# "system" sets the AI's role; "user" is your question
response = client.chat.completions.create(
    model=MODEL,  # gpt-5.6-luna: good value, fast
    messages=[
        {"role": "system", "content": "You are a professional business assistant skilled at writing formal emails."},
        {"role": "user", "content": "Write an email to a client telling them our new product launch event is next Friday at 3 PM, and invite them to attend."}
    ],
    temperature=0.7  # creativity: 0 = conservative, 1 = creative
)

# Extract the AI's reply
ai_reply = response.choices[0].message.content
print("=" * 50)
print("Email written by AI:")
print("=" * 50)
print(ai_reply)

Email written by AI:
**Subject:** Invitation to Our New Product Launch Event  

Dear [Client's Name],  

I hope this message finds you well.  

We are excited to announce the official launch of our newest product, [Product Name], and we would be honored to have you join us for this special occasion. The event will take place next Friday, [Date], at 3:00 PM at [Venue/Address or Virtual Platform].  

This launch marks a significant milestone for us, and we would love to share the story behind the product, its key features, and how it can bring value to you and your business. It will also be a great opportunity to connect and celebrate this achievement together.  

Please let us know if you’ll be able to attend by replying to this email or contacting [Contact Person] at [Phone Number/Email]. We’d be happy to provide additional details, directions, or any accommodations you may need.  

We truly value your support and hope to see you there!  

Warm regards,  
[Your Full Name]  
[Your Job T

### Discussion

Look at the email AI wrote and think about:
- Is the format professional? Is the tone appropriate?
- Is any important information missing?
- If you could change something, what would you edit?

> Try modifying the text inside `"content"` above to have AI write different content!

---

## Activity 2: Compare AI Responses with Different "Roles"

### Activity Goal
Same question, different "roles" for the AI — observe how the response style changes. Understand the power of the `system` message.

In [10]:
# Activity 2: Same question, three different roles

question = "What is artificial intelligence?"

# Role 1: University Professor
print("=" * 50)
print("[Role 1: University Professor]")
print("=" * 50)
response1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a computer science professor. Answer in a rigorous, academic manner."},
        {"role": "user", "content": question}
    ],
    temperature=0.5
)
print(response1.choices[0].message.content)

print()

# Role 2: Elementary School Teacher
print("=" * 50)
print("[Role 2: Elementary School Teacher]")
print("=" * 50)
response2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an elementary school teacher. Explain concepts in simple, fun language that a 10-year-old can understand."},
        {"role": "user", "content": question}
    ],
    temperature=0.7
)
print(response2.choices[0].message.content)

print()

# Role 3: Stand-up Comedian
print("=" * 50)
print("[Role 3: Stand-up Comedian]")
print("=" * 50)
response3 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a stand-up comedian. Answer with humor and wit, throwing in some jokes."},
        {"role": "user", "content": question}
    ],
    temperature=1.2  # higher temperature = more creative
)
print(response3.choices[0].message.content)

[Role 1: University Professor]
As a field of computer science, artificial intelligence (AI) is most precisely understood as **the study and design of rational agents**—systems that perceive their environment, reason about the implications of those perceptions, and take actions that maximize their expected chance of achieving a specified goal. This definition, formalized by Stuart Russell and Peter Norvig in their canonical textbook *Artificial Intelligence: A Modern Approach*, provides a rigorous frame that distinguishes AI from mere automation or algorithm design. Rather than focusing on imitating human behavior per se, this "rational agent" view anchors AI in the engineering principle of optimal goal-seeking under uncertainty.

Historically, the field has been shaped by two intellectual traditions. The **symbolic paradigm** (dominant from the mid-1950s and through the 1980s) treated intelligence as explicit manipulation of logical symbols, exemplified by expert systems that encoded h

### Discussion

- How do the three responses differ? Which style is best for learning?
- Can you use this same role-setting technique in the ChatGPT web interface?
- Can you think of other interesting "roles"? Try them in the code above!

---

## Activity 3: Exploring AI's Capability Boundaries

### Activity Goal
Purposefully ask AI some "hard questions" to observe its limitations. Understand what AI can and cannot do.

In [11]:
# Activity 3: Testing AI's capability boundaries

# Test 1: Ask about very recent events (AI training data has a cutoff date)
print("[Test 1: Timeliness]")
print("Question: What AI policy did China release in June 2025?")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What major AI policy did China release in June 2025? Please elaborate."}
    ]
)
print(response.choices[0].message.content)
print()

# Test 2: Ask a precise calculation question
print("[Test 2: Math Calculation]")
print("Question: 12345 * 67890 = ?")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Calculate 12345 * 67890 mentally. Do not use code, just give the answer."}
    ]
)
print(response.choices[0].message.content)
print("Hint: The actual answer is 838,102,050 — AI may get it right or wrong!")
print()

# Test 3: Push AI to say "I don't know"
print('[Test 3: Making AI say "I don\'t know"]')
print("Question: Describe in detail the complete life story of a fictional person named John Doe Smith.")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'user', 'content': 'Describe in detail the complete life story of a fictional person named John Doe Smith. If this person is made up, just say you don\'t know.'}
    ]
)
print(response.choices[0].message.content)

[Test 1: Timeliness]
Question: What AI policy did China release in June 2025?
In June 2025, China released a major policy document titled **“Action Plan for Artificial Intelligence Plus”** (often referred to as the **“AI+ Action Plan”**, 人工智能+行动方案), issued by the State Council. This policy is a cornerstone for China’s next stage of AI development, aiming to integrate AI deeply into the real economy and society.

### Key Elements of the Policy:

1. **Goal of “AI+” Integration**  
   The plan promotes the wide application of AI across industries such as manufacturing, agriculture, finance, healthcare, education, transportation, energy, and public services. It encourages enterprises and institutions to adopt AI to upgrade productivity, improve efficiency, and reshape traditional industries.

2. **Phased Targets**  
   - By **2027**, AI is expected to be deeply integrated into key sectors of the economy and society, with a sharp rise in the share of AI-driven industrial growth.  
   - By *

### Discussion

- Which test did AI perform best on? Which was worst?
- Timeliness: does AI know recent events? Why or why not?
- Math: is AI truly "calculating" or "guessing"?
- **Key insight**: AI is a "text generator", not a "knowledge base". Its goal is to produce plausible-looking text, not necessarily factually correct text.

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| OpenAI API call | Send messages to GPT via code and get replies |
| Role setting | Control AI's response style via `system` messages |
| Testing AI boundaries | Understand AI's limitations in timeliness, math, etc. |
| Model comparison | Get a feel for how different parameters affect output |

### Homework
1. Modify the email in Activity 1 — ask AI to write different types (job application, thank-you letter, etc.)
2. Create your own "role" and experiment with role-play conversations
3. Search for "creative ChatGPT role prompts" and collect 3 prompts you find interesting